In [1]:
import pandas as pd

df = pd.read_csv("combined_financial_analysis.csv")


In [2]:
df["Date"] = pd.to_datetime(df["Date"])

In [3]:
feature_cols = [
    col for col in df.columns 
    if col not in ["Date", "Ticker", "Stock_Price"]
]

In [4]:
missing_percent = df.isna().mean() * 100

cols_to_drop = missing_percent[missing_percent > 50].index


In [5]:
print ( cols_to_drop)

Index(['Inventory_Turnover', 'Working_Capital'], dtype='object')


In [6]:

df = df.drop(columns=cols_to_drop)

In [7]:
missing_df = (df.isna().mean()
                  .mul(100)
                  .reset_index()
            )

missing_df.columns = ["column", "missing_percent"]

missing_df = missing_df.sort_values(
    by="missing_percent",
    ascending=False
)

print(missing_df)

                         column  missing_percent
15                          ROE        36.809816
16                          ROA        36.809816
29            Assets_Growth_YoY        36.809816
28        Net_Income_Growth_YoY        36.809816
27           Revenue_Growth_YoY        36.809816
26  Operating_Income_Growth_QoQ        36.809816
25               EPS_Growth_QoQ        36.809816
24            Equity_Growth_QoQ        36.809816
23            Assets_Growth_QoQ        36.809816
22        Net_Income_Growth_QoQ        36.809816
21           Revenue_Growth_QoQ        36.809816
20                EBITDA_Margin        36.809816
19             Operating_Margin        36.809816
18          Gross_Profit_Margin        36.809816
17            Net_Profit_Margin        36.809816
30            Equity_Growth_YoY        36.809816
12         Fixed_Asset_Turnover        34.969325
11     Working_Capital_Turnover        34.969325
10            Payables_Turnover        34.969325
9          Receivabl

In [8]:

feature_cols = [ col for col in df.columns if col not in ["Date", "Ticker", "Stock_Price","Sector"]]


In [9]:
for col in feature_cols:
    sector_median = df.groupby("Sector")[col].transform("median")
    global_median = df[col].median()
    
    df[col] = df[col].fillna(sector_median)
    df[col] = df[col].fillna(global_median)
    

In [10]:
#Add missing indicator:
#This lets LSTM learn:Missingness pattern itself is informative.
"""
for col in feature_cols:
    if df[col].isna().sum() > 0:
        df[col + "_missing"] = df[col].isna().astype(int)
"""

'\nfor col in feature_cols:\n    if df[col].isna().sum() > 0:\n        df[col + "_missing"] = df[col].isna().astype(int)\n'

In [11]:
print("Total remaining NaNs:", df.isna().sum().sum())

Total remaining NaNs: 0


In [12]:
import numpy as np

def compute_slope(x):
    return np.polyfit(range(len(x)), x, 1)[0]
    

In [13]:
agg_dict = {}

for col in feature_cols:
    agg_dict[col] = ["mean", "std", "last"]

In [14]:

agg_df = df.groupby("Ticker").agg(agg_dict)


In [15]:
agg_df.columns = [ f"{col[0]}_{col[1]}" for col in agg_df.columns]

agg_df = agg_df.reset_index()

In [16]:
print (  agg_df)

   Ticker  Current_Ratio_mean  Current_Ratio_std  Current_Ratio_last  \
0    AAPL            0.900296           0.052576            0.973745   
1    ABNB            1.477999           0.206588            1.377170   
2    ADBE            1.056535           0.065619            0.996373   
3     ADI            2.031974           0.200133            2.189925   
4     ADP            1.020952           0.023872            1.032306   
..    ...                 ...                ...                 ...   
94   WDAY            1.835388           0.000000            1.835388   
95    WDC            1.611965           0.387446            1.454985   
96    WMT            0.815468           0.028668            0.802889   
97    XEL            0.686165           0.000000            0.686165   
98     ZS            1.427798           0.386986            1.824131   

    Quick_Ratio_mean  Quick_Ratio_std  Quick_Ratio_last  Cash_Ratio_mean  \
0           0.858330         0.053694          0.937561    

In [17]:
import numpy as np

def add_trend(group):
    for col in feature_cols:
        if group[col].notna().sum() > 1:
            x = np.arange(len(group))
            y = group[col].values
            slope = np.polyfit(x, y, 1)[0]
            group[f"{col}_trend"] = slope
        else:
            group[f"{col}_trend"] = 0
    return group.iloc[-1:]  # keep one row per ticker

trend_df = df.sort_values("Date").groupby("Ticker").apply(add_trend)
trend_df = trend_df.reset_index(drop=True)

C:\Users\radha\AppData\Local\Temp\ipykernel_24548\2638287432.py:14: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  trend_df = df.sort_values("Date").groupby("Ticker").apply(add_trend)


In [18]:
trend_df.columns

Index(['Date', 'Stock_Price', 'Current_Ratio', 'Quick_Ratio', 'Cash_Ratio',
       'Debt_to_Equity', 'Debt_to_Assets', 'Equity_Ratio', 'Asset_Turnover',
       'Receivables_Turnover', 'Payables_Turnover', 'Working_Capital_Turnover',
       'Fixed_Asset_Turnover', 'Sector', 'Ticker', 'ROE', 'ROA',
       'Net_Profit_Margin', 'Gross_Profit_Margin', 'Operating_Margin',
       'EBITDA_Margin', 'Revenue_Growth_QoQ', 'Net_Income_Growth_QoQ',
       'Assets_Growth_QoQ', 'Equity_Growth_QoQ', 'EPS_Growth_QoQ',
       'Operating_Income_Growth_QoQ', 'Revenue_Growth_YoY',
       'Net_Income_Growth_YoY', 'Assets_Growth_YoY', 'Equity_Growth_YoY',
       'Current_Ratio_trend', 'Quick_Ratio_trend', 'Cash_Ratio_trend',
       'Debt_to_Equity_trend', 'Debt_to_Assets_trend', 'Equity_Ratio_trend',
       'Asset_Turnover_trend', 'Receivables_Turnover_trend',
       'Payables_Turnover_trend', 'Working_Capital_Turnover_trend',
       'Fixed_Asset_Turnover_trend', 'ROE_trend', 'ROA_trend',
       'Net_Profit_

In [19]:
feature_cols = [ col for col in agg_df.columns if col not in ["Date", "Ticker", "Stock_Price", "Sector"]]

df_filter = agg_df.dropna(subset=feature_cols, how="all")


In [20]:
final_df = agg_df.merge(trend_df, on="Ticker", how="inner")

In [21]:
print(agg_df.shape)
print(trend_df.shape)
print(agg_df["Ticker"].nunique())
print(trend_df["Ticker"].nunique())

(99, 82)
(99, 58)
99
99


In [22]:

print(list(final_df.columns))


['Ticker', 'Current_Ratio_mean', 'Current_Ratio_std', 'Current_Ratio_last', 'Quick_Ratio_mean', 'Quick_Ratio_std', 'Quick_Ratio_last', 'Cash_Ratio_mean', 'Cash_Ratio_std', 'Cash_Ratio_last', 'Debt_to_Equity_mean', 'Debt_to_Equity_std', 'Debt_to_Equity_last', 'Debt_to_Assets_mean', 'Debt_to_Assets_std', 'Debt_to_Assets_last', 'Equity_Ratio_mean', 'Equity_Ratio_std', 'Equity_Ratio_last', 'Asset_Turnover_mean', 'Asset_Turnover_std', 'Asset_Turnover_last', 'Receivables_Turnover_mean', 'Receivables_Turnover_std', 'Receivables_Turnover_last', 'Payables_Turnover_mean', 'Payables_Turnover_std', 'Payables_Turnover_last', 'Working_Capital_Turnover_mean', 'Working_Capital_Turnover_std', 'Working_Capital_Turnover_last', 'Fixed_Asset_Turnover_mean', 'Fixed_Asset_Turnover_std', 'Fixed_Asset_Turnover_last', 'ROE_mean', 'ROE_std', 'ROE_last', 'ROA_mean', 'ROA_std', 'ROA_last', 'Net_Profit_Margin_mean', 'Net_Profit_Margin_std', 'Net_Profit_Margin_last', 'Gross_Profit_Margin_mean', 'Gross_Profit_Margin_

In [23]:
if 'Stock_Price' in final_df.columns:
    print ( "Yes")

Yes


In [24]:
if "EPS_Growth_QoQ_last" in final_df.columns:
    print ( "Yes")

Yes


In [25]:

final_df["growth_score"] = ( 0.4 * final_df["Revenue_Growth_YoY_last"] +
                                0.3 * final_df["EPS_Growth_QoQ_last"] +
                                0.2 * final_df["Net_Income_Growth_YoY_last"] + 
                                0.1 * final_df["Debt_to_Equity_mean"] 
                            )

In [26]:

final_df["rank"] = final_df["growth_score"].rank(ascending=False)


In [28]:
#inclduing rank as well

final_df["growth_rank"] = ( final_df["growth_score"].rank(method="dense", ascending=False).astype(int))

ranked_df = final_df.sort_values("growth_rank")

cols_to_export = [
    "Ticker",
    "growth_score",
    "growth_rank",
    "EPS_Growth_QoQ_last",
    "Revenue_Growth_YoY_last",
    "ROE_last"
]

ranked_df[cols_to_export].to_csv(
    "ranked_growth_stocks_withRank.csv",
    index=False
)

In [ ]:
#Lot of companies have rank as 0 
